In [1]:
import pandas as pd
import networkx as nx

def build_graph_from_edge_list(
    edge_df,
    source_col,
    target_col,
    weight_col
):
    """
    Build an undirected weighted NetworkX graph from an edge list.
    """
    G = nx.Graph()

    for _, row in edge_df.iterrows():
        source = str(row[source_col]).strip()
        target = str(row[target_col]).strip()
        weight = row[weight_col]

        if pd.isna(source) or pd.isna(target) or pd.isna(weight):
            continue

        if source == target:
            continue

        G.add_edge(source, target, weight=float(weight))

    return G


def graph_summary(G, graph_name):
    """
    Compute basic graph-level statistics.
    """
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    density = nx.density(G) if n_nodes > 1 else 0

    connected_components = list(nx.connected_components(G))
    n_components = len(connected_components)

    largest_component_size = (
        max(len(component) for component in connected_components)
        if connected_components
        else 0
    )

    average_degree = (
        sum(dict(G.degree()).values()) / n_nodes
        if n_nodes > 0
        else 0
    )

    average_weighted_degree = (
        sum(dict(G.degree(weight="weight")).values()) / n_nodes
        if n_nodes > 0
        else 0
    )

    return {
        "graph": graph_name,
        "nodes": n_nodes,
        "edges": n_edges,
        "density": density,
        "connected_components": n_components,
        "largest_component_size": largest_component_size,
        "average_degree": average_degree,
        "average_weighted_degree": average_weighted_degree,
    }

In [7]:
before_edges = pd.read_csv("company_co_mentions_no_allias.csv")

G_before = build_graph_from_edge_list(
    before_edges,
    source_col="company_1",
    target_col="company_2",
    weight_col="article_count"
)

In [8]:
after_edges = pd.read_csv("company_comention_edges_all_2.csv")

G_after = build_graph_from_edge_list(
    after_edges,
    source_col="source",
    target_col="target",
    weight_col="weight"
)

In [9]:
summary_df = pd.DataFrame([
    graph_summary(G_before, "Before alias matching"),
    graph_summary(G_after, "After alias matching"),
])

summary_df

,graph,nodes,edges,density,connected_components,largest_component_size,average_degree,average_weighted_degree
0,Before alias matching,854,2482,0.006814,65,708,5.812646,13.266979
1,After alias matching,1997,9703,0.004869,31,1934,9.717576,20.273410
